# EDA - Mood vs Tracks (Silver en S3)

Objetivo: analisis exploratorio profundo para evaluar la viabilidad de clasificacion emocional.

Restricciones clave:
- Leer exclusivamente desde la capa Silver en S3.
- No leer CSV originales ni Bronze.
- Cada grafico debe tener interpretacion util.

Este notebook sigue EDA_SPEC.MD (paso 3) y busca conclusiones accionables para fases de modelado posteriores (sin hacer modelado aqui).

In [ ]:
import os
from typing import Dict, List

import boto3
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.fs as fs
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from src.config import load_settings

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

settings = load_settings()
settings

In [ ]:
def _s3_filesystem(settings):
    session = boto3.Session(
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
        aws_session_token=os.getenv("AWS_SESSION_TOKEN"),
        region_name=settings.aws_region,
    )
    creds = session.get_credentials()
    access_key = creds.access_key if creds else None
    secret_key = creds.secret_key if creds else None
    session_token = creds.token if creds else None

    kwargs = {}
    if settings.s3_endpoint_url:
        kwargs["endpoint_override"] = settings.s3_endpoint_url

    return fs.S3FileSystem(
        access_key=access_key,
        secret_key=secret_key,
        session_token=session_token,
        region=settings.aws_region,
        **kwargs,
    )


def load_silver_dataset(dataset_name: str):
    s3 = _s3_filesystem(settings)
    base_path = f"{settings.bucket_name}/silver/{dataset_name}"
    dataset = ds.dataset(base_path, filesystem=s3, format="parquet")
    table = dataset.to_table()
    df = table.to_pandas()
    return df, f"s3://{settings.bucket_name}/silver/{dataset_name}/"


mood_df, mood_path = load_silver_dataset("mood_dataset")
tracks_df, tracks_path = load_silver_dataset("tracks_dataset")

print("Mood path:", mood_path, "rows:", len(mood_df))
print("Tracks path:", tracks_path, "rows:", len(tracks_df))

## 1. Comprension general de los datasets

In [ ]:
MOOD_COLUMN_DESC: Dict[str, str] = {
    "uri": "Identificador unico del track",
    "duration_ms": "Duracion en milisegundos",
    "mood_label": "Clase objetivo numerica",
    "mood_name": "Etiqueta textual de emocion (si existe)",
    "danceability": "Ritmicidad, 0-1",
    "energy": "Intensidad/energia, 0-1",
    "speechiness": "Presencia de voz hablada, 0-1",
    "acousticness": "Acusticidad, 0-1",
    "instrumentalness": "Instrumental, 0-1",
    "liveness": "Sensacion de concierto, 0-1",
    "valence": "Valencia emocional, 0-1",
    "loudness": "Loudness en dB",
    "tempo": "Tempo en BPM",
    "spec_rate": "Ritmo espectral (feature derivada)",
}

TRACKS_COLUMN_DESC: Dict[str, str] = {
    "track_id": "Identificador unico del track",
    "track_name": "Nombre del track",
    "artists": "Artistas",
    "album_name": "Album",
    "track_genre": "Genero",
    "popularity": "Popularidad (0-100)",
    "duration_ms": "Duracion en milisegundos",
    "explicit": "Contenido explicito",
    "danceability": "Ritmicidad, 0-1",
    "energy": "Intensidad/energia, 0-1",
    "speechiness": "Presencia de voz hablada, 0-1",
    "acousticness": "Acusticidad, 0-1",
    "instrumentalness": "Instrumental, 0-1",
    "liveness": "Sensacion de concierto, 0-1",
    "valence": "Valencia emocional, 0-1",
    "loudness": "Loudness en dB",
    "tempo": "Tempo en BPM",
}

def dataset_overview(df: pd.DataFrame, name: str, desc_map: Dict[str, str]):
    display(df.head(5))
    print("Shape", df.shape)
    dtypes = df.dtypes.rename("dtype")
    nulls = df.isna().sum().rename("nulls")
    null_pct = (nulls / len(df) * 100).rename("null_pct")
    sample = df.head(1).T.rename(columns={0: "sample"})
    summary = pd.concat([dtypes, nulls, null_pct, sample], axis=1)
    summary["description"] = summary.index.map(lambda c: desc_map.get(c, "(sin descripcion)"))
    display(summary.sort_values("null_pct", ascending=False))

print("Mood dataset overview")
dataset_overview(mood_df, "mood", MOOD_COLUMN_DESC)

print("Tracks dataset overview")
dataset_overview(tracks_df, "tracks", TRACKS_COLUMN_DESC)

### 1.1 Estadisticas descriptivas completas

Incluye media, mediana, moda, desviacion estandar, minimos, maximos y percentiles para variables numericas.

In [ ]:
def descriptive_stats(df: pd.DataFrame, name: str):
    numeric_cols = df.select_dtypes(include=["number"]).columns
    if len(numeric_cols) == 0:
        print(f"{name}: no hay columnas numericas")
        return
    desc = df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    desc["median"] = df[numeric_cols].median()
    try:
        desc["mode"] = df[numeric_cols].mode().iloc[0]
    except IndexError:
        desc["mode"] = np.nan
    ordered = [
        "count",
        "mean",
        "median",
        "mode",
        "std",
        "min",
        "1%",
        "5%",
        "25%",
        "50%",
        "75%",
        "95%",
        "99%",
        "max",
    ]
    display(desc[ordered].sort_index())
    
    categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns
    if len(categorical_cols):
        display(df[categorical_cols].describe().T)
    

print("Estadisticas descriptivas - Mood")
descriptive_stats(mood_df, "mood")
print("Estadisticas descriptivas - Tracks")
descriptive_stats(tracks_df, "tracks")

### 1.2 Calidad de datos: nulos y cardinalidad

Se priorizan columnas con nulos altos o cardinalidad extrema que pueden afectar el modelado.

In [ ]:
def data_quality_report(df: pd.DataFrame, name: str, null_threshold: float = 0.05):
    null_pct = df.isna().mean().sort_values(ascending=False)
    high_nulls = null_pct[null_pct >= null_threshold]
    print(f"Calidad de datos - {name}")
    if not high_nulls.empty:
        print("Columnas con nulos relevantes:")
        display(high_nulls.rename("null_pct"))
    else:
        print("Sin columnas con nulos relevantes (>= 5%).")
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns
    if len(categorical_cols):
        cardinality = df[categorical_cols].nunique().sort_values(ascending=False)
        display(cardinality.rename("unique_values").head(10))
        print("Interpretacion: cardinalidad alta puede requerir estrategias de codificacion posteriores.")


data_quality_report(mood_df, "mood")
data_quality_report(tracks_df, "tracks")

## 2. Distribucion de la variable objetivo (mood)
Analizamos balance de clases y sus implicaciones para el modelado.

In [ ]:
target_col = "mood_label" if "mood_label" in mood_df.columns else None
if target_col is None:
    raise ValueError("mood_label no existe en el dataset Silver")

target_counts = mood_df[target_col].value_counts(dropna=False).sort_index()
display(target_counts)

plt.figure(figsize=(8, 4))
sns.barplot(x=target_counts.index.astype(str), y=target_counts.values)
plt.title("Distribucion de clases (mood_label)")
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.show()

class_balance = (target_counts / target_counts.sum()).rename("ratio")
display(class_balance)

In [ ]:
def interpret_class_balance(counts: pd.Series):
    total = counts.sum()
    ratios = counts / total
    max_ratio = ratios.max()
    min_ratio = ratios.min()
    imbalance_ratio = max_ratio / max(min_ratio, 1e-12)
    majority_class = ratios.idxmax()
    minority_class = ratios.idxmin()
    entropy = -(ratios * np.log2(ratios + 1e-12)).sum()
    gini = 1 - (ratios**2).sum()
    print("Balance de clases")
    print(f"- Clase mayoritaria: {majority_class} ({max_ratio:.2%})")
    print(f"- Clase minoritaria: {minority_class} ({min_ratio:.2%})")
    print(f"- Ratio desbalance (mayor/minor): {imbalance_ratio:.2f}x")
    print(f"- Entropia de clases: {entropy:.3f} (maximo teorico {np.log2(len(ratios)):.3f})")
    print(f"- Gini de clases: {gini:.3f}")
    if imbalance_ratio >= 3:
        print("Interpretacion: desbalance alto; conviene monitorizar metricas por clase.")
    elif imbalance_ratio >= 1.5:
        print("Interpretacion: desbalance moderado; usar metricas por clase.")
    else:
        print("Interpretacion: clases relativamente balanceadas.")


interpret_class_balance(target_counts)

## 3. Distribucion de features acusticas
Se analizan histogramas y boxplots para detectar sesgo y outliers.

In [ ]:
MOOD_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness", "tempo", "spec_rate"
]
TRACKS_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness", "tempo"
]

def plot_distributions(df: pd.DataFrame, features: List[str], title: str):
    available = [f for f in features if f in df.columns]
    n = len(available)
    if n == 0:
        print("No features disponibles para", title)
        return

    rows = int(np.ceil(n / 3))
    fig, axes = plt.subplots(rows, 3, figsize=(15, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, feat in zip(axes, available):
        sns.histplot(df[feat].dropna(), kde=True, ax=ax)
        ax.set_title(f"{feat} - hist")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(f"{title} - histogramas", y=1.02)
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(rows, 3, figsize=(15, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, feat in zip(axes, available):
        sns.boxplot(x=df[feat], ax=ax)
        ax.set_title(f"{feat} - boxplot")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(f"{title} - boxplots", y=1.02)
    plt.tight_layout()
    plt.show()

plot_distributions(mood_df, MOOD_FEATURES, "Mood dataset")
plot_distributions(tracks_df, TRACKS_FEATURES, "Tracks dataset")

### 3.1 Metricas cuantitativas: asimetria y outliers

Se calcula skewness y porcentaje de outliers por IQR para priorizar transformaciones futuras (sin aplicarlas).

In [ ]:
def skew_outlier_report(df: pd.DataFrame, features: List[str], name: str):
    available = [f for f in features if f in df.columns]
    rows = []
    for feat in available:
        series = df[feat].dropna()
        if series.empty:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_pct = ((series < lower) | (series > upper)).mean() * 100
        rows.append({
            "feature": feat,
            "skew": series.skew(),
            "outlier_pct": outlier_pct,
            "iqr": iqr,
        })
    report = pd.DataFrame(rows).sort_values("outlier_pct", ascending=False)
    print(f"Skew/outliers - {name}")
    display(report)
    if not report.empty:
        high_skew = report[report["skew"].abs() >= 1]
        high_out = report[report["outlier_pct"] >= 5]
        if not high_skew.empty:
            print("Interpretacion: asimetrias fuertes en", list(high_skew["feature"]))
        if not high_out.empty:
            print("Interpretacion: outliers relevantes en", list(high_out["feature"]))
        if high_skew.empty and high_out.empty:
            print("Interpretacion: distribuciones relativamente estables.")
    

skew_outlier_report(mood_df, MOOD_FEATURES, "mood")
skew_outlier_report(tracks_df, TRACKS_FEATURES, "tracks")

## 4. Relacion feature vs mood
Boxplots por clase para ver discriminacion entre emociones.

In [ ]:
def plot_feature_vs_mood(df: pd.DataFrame, features: List[str], target: str):
    available = [f for f in features if f in df.columns]
    n = len(available)
    rows = int(np.ceil(n / 3))
    fig, axes = plt.subplots(rows, 3, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, feat in zip(axes, available):
        sns.boxplot(data=df, x=target, y=feat, ax=ax)
        ax.set_title(f"{feat} por {target}")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_feature_vs_mood(mood_df, MOOD_FEATURES, "mood_label")

### 4.1 Features mas discriminativas respecto al mood

Se estima la varianza explicada por clase ($\eta^2$) para priorizar features con mayor separacion entre emociones.

In [ ]:
def eta_squared_by_feature(df: pd.DataFrame, features: List[str], target: str):
    rows = []
    overall_mean = df[features].mean(numeric_only=True)
    for feat in [f for f in features if f in df.columns]:
        series = df[[feat, target]].dropna()
        if series.empty:
            continue
        group_means = series.groupby(target)[feat].mean()
        group_sizes = series.groupby(target)[feat].size()
        ss_between = ((group_means - overall_mean.get(feat, 0))**2 * group_sizes).sum()
        ss_total = ((series[feat] - series[feat].mean())**2).sum()
        eta2 = float(ss_between / ss_total) if ss_total else np.nan
        rows.append({"feature": feat, "eta_squared": eta2})
    result = pd.DataFrame(rows).sort_values("eta_squared", ascending=False)
    display(result)
    if not result.empty:
        top = result.head(5)
        print("Interpretacion: las features con mayor varianza explicada por mood son:")
        for _, row in top.iterrows():
            print(f"- {row['feature']}: eta^2={row['eta_squared']:.3f}")
    return result

eta2_table = eta_squared_by_feature(mood_df, MOOD_FEATURES, "mood_label")

## 5. Correlacion entre features
Se busca multicolinealidad y redundancias.

In [ ]:
numeric_cols = mood_df.select_dtypes(include=["number"]).columns
corr = mood_df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlacion - Mood dataset")
plt.show()

### 5.1 Correlaciones altas y redundancia

Se listan pares con alta correlacion absoluta para evaluar multicolinealidad y posibles redundancias.

In [ ]:
def high_correlations(df: pd.DataFrame, threshold: float = 0.6):
    corr = df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = (
        upper.stack()
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "corr"})
        .sort_values("corr", ascending=False)
    )
    high = pairs[pairs["corr"] >= threshold]
    display(high.head(15))
    if not high.empty:
        print("Interpretacion: correlaciones altas pueden indicar redundancia o necesidad de regularizacion.")
    else:
        print("Interpretacion: no se observan correlaciones muy altas con el umbral actual.")


high_correlations(mood_df.select_dtypes(include=["number"]))

## 6. Comparativa Mood vs Tracks (dataset shift)
Se comparan distribuciones en features compartidas.

In [ ]:
shared = [f for f in TRACKS_FEATURES if f in mood_df.columns and f in tracks_df.columns]

plot_sample_n = 5000
mood_sample = mood_df.sample(min(len(mood_df), plot_sample_n), random_state=42)
tracks_sample = tracks_df.sample(min(len(tracks_df), plot_sample_n), random_state=42)

for feat in shared:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(mood_sample[feat].dropna(), label="mood")
    sns.kdeplot(tracks_sample[feat].dropna(), label="tracks")
    plt.title(f"Comparativa {feat}")
    plt.legend()
    plt.show()

### 6.1 Metricas de dataset shift

Se cuantifica diferencia de medias estandarizada (Cohen d) para features compartidas.

In [ ]:
def cohen_d(a: pd.Series, b: pd.Series) -> float:
    a = a.dropna()
    b = b.dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    mean_diff = a.mean() - b.mean()
    pooled_std = np.sqrt(((a.var(ddof=1) + b.var(ddof=1)) / 2))
    return float(mean_diff / pooled_std) if pooled_std else np.nan


rows = []
for feat in shared:
    rows.append({
        "feature": feat,
        "cohen_d": cohen_d(mood_df[feat], tracks_df[feat]),
        "mean_mood": mood_df[feat].mean(),
        "mean_tracks": tracks_df[feat].mean(),
    })
shift_table = pd.DataFrame(rows).sort_values("cohen_d", key=lambda s: s.abs(), ascending=False)
display(shift_table)
if not shift_table.empty:
    top = shift_table.iloc[0]
    print("Interpretacion: mayores diferencias de distribucion en", top["feature"], "(d=", f"{top['cohen_d']:.2f}", ")")

## 7. Conclusiones

Las conclusiones se derivan de las metricas anteriores y se presentan en la celda siguiente para asegurar que sean accionables y trazables.

In [ ]:
def build_conclusions():
    lines = []
    ratios = target_counts / target_counts.sum()
    imbalance_ratio = ratios.max() / max(ratios.min(), 1e-12)
    if imbalance_ratio >= 3:
        lines.append("Desbalance de clases alto; evaluar metricas por clase y posible reequilibrio en fase posterior.")
    elif imbalance_ratio >= 1.5:
        lines.append("Desbalance moderado; usar metricas por clase en evaluacion.")
    else:
        lines.append("Clases relativamente balanceadas; impacto menor por desbalance.")
    if "eta2_table" in globals() and not eta2_table.empty:
        top_feats = ", ".join(eta2_table.head(3)["feature"].tolist())
        lines.append(f"Features con mayor poder discriminativo (eta^2): {top_feats}.")
    if "shift_table" in globals() and not shift_table.empty:
        top_shift = shift_table.iloc[0]
        lines.append(f"Mayor evidencia de dataset shift en {top_shift['feature']} (Cohen d={top_shift['cohen_d']:.2f}).")
    lines.append("Revisar correlaciones altas para evitar redundancias en fases de modelado.")
    print("Conclusiones clave")
    for item in lines:
        print("-", item)


build_conclusions()